## Cell 1: Import Libraries

In [30]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

## Cell 2: Loading, Cleaning, and Saving data for the tickers in our universe

In [ ]:
etfs = [
    "XLF",  # Financials
    "XLE",  # Energy
    "XLK",  # Technology
    "XLI",  # Industrials
    "XLP",  # Consumer Staples
    "XLY",  # Consumer Discretionary
    "XLV",  # Health Care
    "XLU",  # Utilities
    "IYT",  # Transportation
    "XRT",  # Retail
    "SMH",  # Semiconductors
    "IYR",  # Real Estate
    "IGV",  # Software/Internet
    "XLB",  # Materials
    "VOX",  # Communication Services
]
START_DATE = "2018-01-01"
END_DATE = "2023-12-31"
WIKI_SP500_2018_URL = "https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&oldid=820572003"

# S&P 500 tickers (grab from Wikipedia)
sp500 = pd.read_html(
    WIKI_SP500_2018_URL,
    storage_options={"User-Agent": "Mozilla/5.0"},
)[0]
sp500.rename(columns={"Ticker symbol": "Symbol"}, inplace=True)

# yfinance uses '-' not '.' in tickers
sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)

stock_tickers = sp500["Symbol"].tolist()

all_tickers = stock_tickers + etfs

print(
    f"Requesting data for {len(stock_tickers)} stock and {len(etfs)} etf tickers from Yahoo Finance..."
)

raw_data = yf.download(
    tickers=all_tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    threads=True,
    progress=False,
)
prices = raw_data["Close"]

print(f"Downloaded: {len(prices.columns)} / {len(all_tickers)}")

# Keep stocks with atleast 95% of data
prices = prices.dropna(axis=1, thresh=int(0.95 * len(prices)))
prices = prices.ffill()
prices = prices.dropna()

print(f"Remaining: {len(prices.columns)} / {len(all_tickers)}")

# Calculate log returns
returns = np.log(prices / prices.shift(1)).dropna()

# Separate etfs from sp500
etf_cols = [c for c in prices.columns if c in etfs]
stock_cols = [c for c in prices.columns if c not in etfs]

etf_prices = prices[etf_cols]
stock_prices = prices[stock_cols]
etf_returns = returns[etf_cols]
stock_returns = returns[stock_cols]

# Add to data directory
os.makedirs("../data", exist_ok=True)
etf_prices.to_csv("../data/etf_prices.csv")
stock_prices.to_csv("../data/stock_prices.csv")
etf_returns.to_csv("../data/etf_returns_log.csv")
stock_returns.to_csv("../data/stock_returns_log.csv")

print("Successfully saved prices and returns data.")
print(f"ETF prices:   {etf_prices.shape}")
print(f"Stock prices: {stock_prices.shape}")
print(f"ETF returns:   {etf_returns.shape}")
print(f"Stock returns: {stock_returns.shape}")


[                       1%                       ]  3 of 520 completed

Requesting data for 505 stock and 15 etf tickers from Yahoo Finance...


[*                      2%                       ]  10 of 520 completed$HCN: possibly delisted; no price data found  (1d 2018-01-01 -> 2023-12-31)
[**                     4%                       ]  22 of 520 completed$RE: possibly delisted; no timezone found
[**                     5%                       ]  24 of 520 completed$HES: possibly delisted; no timezone found
[**                     5%                       ]  25 of 520 completed$XEC: possibly delisted; no timezone found
[**                     5%                       ]  27 of 520 completed$DRE: possibly delisted; no timezone found
[***                    7%                       ]  38 of 520 completed$CBS: possibly delisted; no timezone found
$LB: possibly delisted; no price data found  (1d 2018-01-01 -> 2023-12-31) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1703998800")
[****                   8%                       ]  40 of 520 completed$DPS: possibly delisted; no price data found  (1d 20

Downloaded: 520 / 520
Remaining: 425 / 520
Successfully saved prices and returns data.
ETF prices:   (1509, 15)
Stock prices: (1509, 410)
ETF returns:   (1508, 15)
Stock returns: (1508, 410)


## Cell 3: Creating Metadata Table

In [ ]:
etfs = [
    "XLE",  # Energy
    "XLF",  # Financial
    "XLK",  # Technology
]

In [97]:
metadata = sp500[["Symbol", "Security", "GICS Sector", "GICS Sub Industry"]].copy()


def map_etf_and_sector(row):
    sector = row["GICS Sector"]
    sub_ind = row["GICS Sub Industry"]

    # 1. Specialized Tech (IGV & SMH)
    if sub_ind in [
        "Application Software",
        "Systems Software",
        "Home Entertainment Software",
        "Internet Services & Infrastructure",
    ]:
        return pd.Series(["Software/Internet", "IGV"])
    if sub_ind in [
        "Semiconductors",
        "Semiconductor Materials & Equipment",
        "Semiconductor Equipment",
    ]:
        return pd.Series(["Semiconductors", "SMH"])

    # 2. Specialized Industrials/Consumer (IYT & XRT)
    if sub_ind in [
        "Air Freight & Logistics",
        "Railroads",
        "Rail Transportation",
        "Airlines",
        "Passenger Airlines",
        "Trucking",
        "Cargo Ground Transportation",
    ]:
        return pd.Series(["Transportation", "IYT"])
    if sub_ind in [
        "Broadline Retail",
        "Automotive Retail",
        "Computer & Electronics Retail",
        "Home Improvement Retail",
        "Apparel Retail",
        "Specialty Stores",
        "Department Stores",
    ]:
        return pd.Series(["Retail", "XRT"])

    # 3. Broad GICS Sector Mapping
    mapping = {
        "Financials": ["Financials", "XLF"],
        "Energy": ["Energy", "XLE"],
        "Information Technology": ["Information Technology", "XLK"],
        "Industrials": ["Industrials", "XLI"],
        "Consumer Staples": ["Consumer Staples", "XLP"],
        "Consumer Discretionary": ["Consumer Discretionary", "XLY"],
        "Health Care": ["Health Care", "XLV"],
        "Utilities": ["Utilities", "XLU"],
        "Real Estate": ["Real Estate", "IYR"],
        "Materials": ["Materials", "XLB"],
        "Communication Services": ["Communication Services", "XLC"],  # or VOX
    }
    if sector in mapping:
        return pd.Series(mapping[sector])

    return pd.Series([None, None])


# Map to our universe of sectors/etfs
metadata[["sector", "etf"]] = metadata.apply(map_etf_and_sector, axis=1)

metadata = metadata.drop(columns=["GICS Sector", "GICS Sub Industry"])

metadata.set_index("Symbol", inplace=True)

# Save metadata
metadata.to_csv("../data/stock_meta.csv", index=True)

# Print the count by readable Sector Name
print("\n--- Number of Stocks per Target Sector ---")
display(metadata["sector"].value_counts().to_frame(name="Count"))



--- Number of Stocks per Target Sector ---


,Count
sector,
Financials,68
Consumer Discretionary,66
Health Care,61
Industrials,53
Information Technology,43
Consumer Staples,34
Real Estate,33
Energy,32
Utilities,28
